# Card testing: what your detector can see

This self-contained defensive lab uses invented identifiers and synthetic events. It makes no network requests to a payment system and contains no card numbers.

**Run in Google Colab:** upload this notebook to [Colab](https://colab.research.google.com/), choose a Python CPU runtime, then **Runtime → Run all**. The first cell installs pinned libraries when needed. No separate `.py` or CSV upload is required. A local Jupyter runtime uses the same cells; connect it only if you intentionally want this notebook to execute on your computer.

**Run locally without Colab:** use Python 3.12 or 3.13, install `requirements.txt` in a virtual environment, and run `python card_testing_lab.py --output-dir lab_output`.

The model learns an intentionally narrow synthetic classic-fraud pattern: amount, hour, distance, and invented prior card activity. We then evaluate different synthetic checkout behavior. This is an experiment about detector scope, not a benchmark of real fraud systems or a replication of the Day 61 results.

## Interpret the numbers carefully

- Classic-fraud precision, recall and false-positive rate come from a disjoint held-out test set. A separate validation set chooses the model threshold. No SMOTE or class weights are used.
- The transaction model scores **payments only**. Card saves are explicitly unscored, not classified as legitimate. The all-event coverage table counts them as uncovered; a separate payment-only table shows model coverage on scored attack events.
- Rules use fixed UTC five-minute bins. They retrospectively mark events in a qualifying bin. These counts do not establish when an online detector would alert or how many attempts it would prevent.
- Genuine background events overlap the attack windows. Genuine shared-device purchases and a genuine card-save burst deliberately test false positives.
- All exported cards are different invented IDs. `prior_card_attempts_1h` is invented historical context, not calculated from the exported four-hour CSV; those earlier attempts are omitted from this export.


In [ ]:
# Install only missing/mismatched pinned dependencies in this notebook's runtime.
import importlib.metadata as md
from pathlib import Path
import subprocess
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Use Python 3.12 or 3.13; the verified environment used Python 3.12.")
if sys.version_info >= (3, 14):
    print("Warning: pandas 2.2.3 has no prebuilt wheels for Python 3.14+, so pip may build it from source (slow).")

REQUIREMENTS = '# Tested with Python 3.12. No GPU required.\nnumpy==2.3.5\npandas==2.2.3\nscikit-learn==1.9.1\nscipy==1.18.1\nmatplotlib==3.11.2\n'
Path("card_lab_requirements.txt").write_text(REQUIREMENTS)
required = [line.strip() for line in REQUIREMENTS.splitlines() if line.strip() and not line.startswith("#")]
needs_install = []
for item in required:
    package, version = item.split("==")
    try:
        installed = md.version(package)
    except md.PackageNotFoundError:
        installed = None
    if installed != version:
        needs_install.append(item)
if needs_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--timeout", "120", "-r", "card_lab_requirements.txt"], check=True)
print("Dependencies ready. The lab runs in a fresh process to avoid stale imports.")


## Run the complete experiment

The next cell contains the exact standalone source. Keep the seed and 80 devices for the published baseline. To experiment, change `SPREAD_DEVICES`, rerun, and compare the resulting metrics. A variant does not overwrite the baseline directory. The baseline chart is produced only for 80 devices to avoid labeling a variant as the baseline.


In [ ]:
SOURCE = '#!/usr/bin/env python3\n"""Defensive, synthetic card-testing detector comparison. No network calls or card data.\n\nRun: python card_testing_lab.py --output-dir lab_output\nThe notebook embeds this same file and runs it without uploading any other file.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nos.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mpl_cache"))\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score\nfrom sklearn.model_selection import train_test_split\n\nSEED = 61027\nFEATURES = ["amount_minor", "hour_utc", "distance_km", "prior_card_attempts_1h"]\nDETECTORS = ["toy_model", "device_rule", "checkout_rule", "decline_rule"]\nSCENARIOS = ["legitimate_baseline", "legitimate_shared_device", "legitimate_setup_burst", "loud", "spread"]\nBASE = int(pd.Timestamp("2026-09-26T10:00:00Z").timestamp())\n\n# Counting-rule criteria, shared by apply_detectors() and reference_rule_flags() so the\n# two Python implementations cannot drift apart. The SPL in splunk/ mirrors them by design.\nWINDOW_SECONDS = 300\nRULE_MIN_EVENTS = 20\nRULE_MIN_CARDS = 15\nRULE_MIN_DECLINE_SHARE = .8\nSMALL_PAYMENT_MAX_MINOR = 100\n\n\ndef classic_population(rng: np.random.Generator, n: int, amount_loc: float, amount_scale: float,\n                       hour_weights: list[int], distance_scale: float, prior_rate: float, label: int) -> pd.DataFrame:\n    """One synthetic transaction population; the parameter gap is the classic-fraud assumption."""\n    return pd.DataFrame({\n        "amount_minor": np.clip(rng.lognormal(amount_loc, amount_scale, n), 10, 300000).round(),\n        "hour_utc": rng.choice(np.arange(24), n, p=np.array(hour_weights) / sum(hour_weights)),\n        "distance_km": rng.exponential(distance_scale, n),\n        "prior_card_attempts_1h": rng.poisson(prior_rate, n),\n        "label": label,\n    })\n\n\ndef classic_data(seed: int) -> pd.DataFrame:\n    """Intentionally simplified, overlapping synthetic transaction populations.\n\n    This encodes a classic-fraud assumption; it is not an independent empirical\n    benchmark. No trained model chooses how evaluation card-testing data is made.\n    """\n    rng = np.random.default_rng(seed)\n    benign = classic_population(rng, 12000, 7.7, 1.15, [1]*6 + [3]*3 + [8]*12 + [3]*3, 35, .7, 0)\n    fraud = classic_population(rng, 600, 10.4, .85, [12]*6 + [2]*18, 300, 3.0, 1)\n    df = pd.concat([benign, fraud], ignore_index=True)\n    df.insert(0, "sample_id", [f"classic_{i:06d}" for i in range(len(df))])\n    return df\n\n\ndef train_model(seed: int):\n    df = classic_data(seed)\n    train, holdout = train_test_split(df, test_size=.4, stratify=df.label, random_state=seed)\n    valid, test = train_test_split(holdout, test_size=.5, stratify=holdout.label, random_state=seed + 1)\n    sets = [set(d.sample_id) for d in (train, valid, test)]\n    assert not any(sets[i] & sets[j] for i in range(3) for j in range(i + 1, 3))\n    assert len(set.union(*sets)) == len(df)\n    assert not ({"label", "scenario", "sample_id", "event_id"} & set(FEATURES))\n    model = RandomForestClassifier(n_estimators=180, max_depth=8, min_samples_leaf=6,\n                                   random_state=seed, n_jobs=1)\n    model.fit(train[FEATURES], train.label)\n    valid_scores = model.predict_proba(valid[FEATURES])[:, 1]\n    thresholds = np.arange(.05, .951, .01)\n    quality = [f1_score(valid.label, valid_scores >= t, zero_division=0) for t in thresholds]\n    threshold = float(thresholds[int(np.argmax(quality))])  # ties use lower threshold\n    prediction = model.predict_proba(test[FEATURES])[:, 1] >= threshold\n    tn, fp, fn, tp = confusion_matrix(test.label, prediction, labels=[0, 1]).ravel()\n    metrics = {\n        "threshold": round(threshold, 8), "threshold_selection": "maximize validation F1; no test tuning",\n        "precision": float(precision_score(test.label, prediction, zero_division=0)),\n        "recall": float(recall_score(test.label, prediction, zero_division=0)),\n        "false_positive_rate": float(fp / (tn + fp)),\n        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),\n        "train_rows": len(train), "validation_rows": len(valid), "test_rows": len(test),\n        "training_fraud_rows": int(train.label.sum()), "validation_fraud_rows": int(valid.label.sum()),\n        "test_fraud_rows": int(test.label.sum()), "features": FEATURES,\n        "class_weight": None, "resampling": "none",\n    }\n    split_manifest = pd.concat([d[["sample_id", "label"]].assign(split=name)\n                               for d, name in [(train, "train"), (valid, "validation"), (test, "test")]])\n    return model, threshold, metrics, split_manifest.sort_values("sample_id")\n\n\ndef generate_events(seed: int, spread_devices: int = 80) -> pd.DataFrame:\n    # prior_card_attempts_1h is invented historical context, not reconstructed\n    # from this export; those earlier attempts are omitted. Every exported card is distinct.\n    if not 1 <= spread_devices <= 360:\n        raise ValueError("spread_devices must be between 1 and 360")\n    rng = np.random.default_rng(seed + 100)\n    records = []\n\n    def add(offset, scenario, flow, card, device, outcome, amount, distance=10.0, prior=0):\n        epoch = BASE + int(offset)\n        records.append({"epoch": epoch, "flow": flow, "checkout_id": "shop_demo",\n                        "event_source": "synthetic_lab", "outcome": outcome, "currency": "USD",\n                        "amount_minor": int(amount), "card_fp": f"fake_card_{card}",\n                        "device_fp": f"fake_device_{device}", "distance_km": round(float(distance), 3),\n                        "prior_card_attempts_1h": int(prior), "scenario": scenario,\n                        "label": int(scenario in ("loud", "spread"))})\n\n    # Ordinary background: payments and sparse card saves; no real account data.\n    for i in range(1440):\n        flow = "setup_card" if rng.random() < .08 else "payment"\n        amount = 0 if flow == "setup_card" else (int(rng.integers(10, 101)) if rng.random() < .08\n                                                    else int(np.clip(rng.lognormal(7.7, 1.05), 150, 100000)))\n        add(rng.integers(0, 14400), "legitimate_baseline", flow, f"b{i:05d}", f"b{i:05d}",\n            "approved" if rng.random() < .96 else "declined", amount, rng.exponential(30), rng.poisson(.5))\n\n    # A genuine shared terminal: many different cards, normal purchase amounts.\n    for i in range(24):\n        add(8100 + i * 12, "legitimate_shared_device", "payment", f"shared{i:03d}", "shared_terminal",\n            "approved", 1800 + i * 100, 5, 0)\n\n    # A genuine card-save burst across individual devices. A deliberate FP stress case.\n    for i in range(28):\n        add(12600 + i * 10, "legitimate_setup_burst", "setup_card", f"onboard{i:03d}", f"onboard{i:03d}",\n            "approved", 0, 10, 0)\n\n    # Loud pattern: 120 attempts, one device, 20 minutes, mostly approvals.\n    for i in range(120):\n        flow = "setup_card" if i % 3 == 0 else "payment"\n        add(1800 + i * 10, "loud", flow, f"loud{i:03d}", "loud", "declined" if i % 10 == 0 else "approved",\n            0 if flow == "setup_card" else int(rng.integers(10, 101)), rng.uniform(1, 30), 0)\n\n    # Spread pattern: random arrivals over three hours, one invented card per event.\n    offsets = rng.integers(3600, 14400, 360)\n    for i, offset in enumerate(offsets):\n        flow = "setup_card" if i % 3 == 0 else "payment"\n        add(offset, "spread", flow, f"spread{i:03d}", f"spread{i % spread_devices:03d}",\n            "declined" if i % 10 == 0 else "approved", 0 if flow == "setup_card" else int(rng.integers(10, 101)),\n            rng.uniform(1, 30), 0)\n\n    df = pd.DataFrame(records).sort_values(["epoch", "card_fp"], kind="stable").reset_index(drop=True)\n    df.insert(0, "event_id", [f"evt_{i:06d}" for i in range(len(df))])\n    timestamps = pd.to_datetime(df.epoch, unit="s", utc=True)\n    df.insert(1, "ts", timestamps.dt.strftime("%Y-%m-%dT%H:%M:%SZ"))\n    df["hour_utc"] = timestamps.dt.hour\n    df["bucket_epoch"] = (df.epoch // WINDOW_SECONDS) * WINDOW_SECONDS\n    assert df.event_id.is_unique and df.card_fp.is_unique\n    assert df.epoch.between(BASE, BASE + 14400 - 1).all()\n    assert (pd.to_datetime(df.ts, utc=True).astype("int64") // 10**9 == df.epoch).all()\n    assert (df.loc[df.flow == "setup_card", "amount_minor"] == 0).all()\n    return df\n\n\ndef apply_detectors(df, model, threshold):\n    df = df.copy()\n    df["toy_model_scored"] = (df.flow == "payment").astype(int)\n    df["toy_model_score"] = np.nan\n    scored = df.toy_model_scored == 1\n    df.loc[scored, "toy_model_score"] = model.predict_proba(df.loc[scored, FEATURES])[:, 1]\n    df["toy_model"] = (df.toy_model_score >= threshold).astype(int)\n    device_group = df.groupby(["checkout_id", "device_fp", "bucket_epoch"], sort=True)\n    df["device_attempts"] = device_group.event_id.transform("size")\n    df["device_cards"] = device_group.card_fp.transform("nunique")\n    df["device_declines"] = device_group.outcome.transform(lambda s: (s == "declined").sum())\n    df["device_rule"] = ((df.device_attempts >= RULE_MIN_EVENTS) & (df.device_cards >= RULE_MIN_CARDS)).astype(int)\n    df["decline_rule"] = ((df.device_attempts >= RULE_MIN_EVENTS) & (df.device_cards >= RULE_MIN_CARDS)\n                          & (df.device_declines / df.device_attempts >= RULE_MIN_DECLINE_SHARE)).astype(int)\n    df["candidate"] = ((df.flow == "setup_card") | ((df.flow == "payment") & (df.currency == "USD")\n                                                   & (df.amount_minor <= SMALL_PAYMENT_MAX_MINOR))).astype(int)\n    selected = df.loc[df.candidate == 1].groupby(["checkout_id", "bucket_epoch"], sort=True)\n    counts = selected.agg(checkout_candidates=("event_id", "size"), checkout_cards=("card_fp", "nunique"))\n    df = df.join(counts, on=["checkout_id", "bucket_epoch"])\n    df[["checkout_candidates", "checkout_cards"]] = df[["checkout_candidates", "checkout_cards"]].fillna(0).astype(int)\n    df["checkout_rule"] = ((df.candidate == 1) & (df.checkout_candidates >= RULE_MIN_EVENTS)\n                           & (df.checkout_cards >= RULE_MIN_CARDS)).astype(int)\n    return df\n\n\ndef reference_rule_flags(df):\n    """Independent plain-Python aggregation mirrors SPL; checks pandas implementation."""\n    dev, shop = {}, {}\n    for r in df.to_dict("records"):\n        bucket = int(r["epoch"]) // WINDOW_SECONDS * WINDOW_SECONDS\n        key = (r["checkout_id"], r["device_fp"], bucket)\n        dev.setdefault(key, []).append(r)\n        candidate = (r["flow"] == "setup_card" or (r["flow"] == "payment" and r["currency"] == "USD"\n                                                   and r["amount_minor"] <= SMALL_PAYMENT_MAX_MINOR))\n        if candidate:\n            shop.setdefault((r["checkout_id"], bucket), []).append(r)\n    expected = {k: set() for k in ["device_rule", "checkout_rule", "decline_rule"]}\n    for rows in dev.values():\n        if len(rows) >= RULE_MIN_EVENTS and len({r["card_fp"] for r in rows}) >= RULE_MIN_CARDS:\n            expected["device_rule"].update(r["event_id"] for r in rows)\n            if sum(r["outcome"] == "declined" for r in rows) / len(rows) >= RULE_MIN_DECLINE_SHARE:\n                expected["decline_rule"].update(r["event_id"] for r in rows)\n    for rows in shop.values():\n        if len(rows) >= RULE_MIN_EVENTS and len({r["card_fp"] for r in rows}) >= RULE_MIN_CARDS:\n            expected["checkout_rule"].update(r["event_id"] for r in rows)\n    for rule, ids in expected.items():\n        assert ids == set(df.loc[df[rule] == 1, "event_id"]), f"Independent rule mismatch: {rule}"\n\n\ndef summarize(df):\n    rows = []\n    for scenario in SCENARIOS:\n        subset = df[df.scenario == scenario]\n        payments, setups = subset.flow == "payment", subset.flow == "setup_card"\n        for detector in DETECTORS:\n            flagged = subset[subset[detector] == 1]\n            flagged_payments, flagged_setups = flagged.flow == "payment", flagged.flow == "setup_card"\n            keys = ["checkout_id", "bucket_epoch"] + (["device_fp"] if detector in ("device_rule", "decline_rule") else [])\n            rows.append({"scenario": scenario, "detector": detector, "events": len(subset),\n                         "flagged_events": len(flagged), "coverage": len(flagged) / len(subset),\n                         "flagged_group_intersections": len(flagged[keys].drop_duplicates()),\n                         "payment_events": int(payments.sum()),\n                         "setup_events": int(setups.sum()),\n                         "flagged_payment_events": int(flagged_payments.sum()),\n                         "flagged_setup_events": int(flagged_setups.sum()),\n                         "payment_coverage": int(flagged_payments.sum()) / int(payments.sum()) if payments.any() else None,\n                         "model_unscored_events": int((subset.toy_model_scored == 0).sum()) if detector == "toy_model" else 0})\n    return pd.DataFrame(rows)\n\n\ndef draw_chart(path, summary, classic):\n    plt.rcParams.update({"font.family": "DejaVu Sans", "text.color": "#F5F4FF"})\n    bg, panel, green = "#1a1a2e", "#25253e", "#00B894"\n    fig = plt.figure(figsize=(10.8, 13.5), dpi=100, facecolor=bg)\n    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 1080); ax.set_ylim(1350, 0); ax.axis("off")\n    ax.text(64, 76, "CHECKOUT SECURITY  /  SYNTHETIC LAB", fontsize=14, weight="bold", color="#ABA2FF")\n    ax.text(64, 148, "Good fraud scores.\\nDifferent attack signal.", fontsize=38, weight="bold", linespacing=1.15, va="top")\n    ax.text(64, 295, "A toy transaction model meets card testing.", fontsize=20, color="#D1D0DF")\n    ax.add_patch(FancyBboxPatch((64, 340), 952, 180, boxstyle="round,pad=0,rounding_size=20", facecolor=panel, edgecolor="none"))\n    ax.text(94, 382, "CLASSIC SYNTHETIC FRAUD · HELD-OUT TEST", fontsize=14, weight="bold", color="#BDB7F2")\n    for x, value, caption in [(94, classic["recall"], "recall"), (408, classic["precision"], "precision"), (722, classic["false_positive_rate"], "false-positive rate")]:\n        ax.text(x, 448, f"{100*value:.1f}%", fontsize=35, weight="bold", color=green)\n        ax.text(x, 488, caption, fontsize=15, color="#D1D0DF")\n    ax.text(64, 578, "ATTACK EVENTS COVERED BY EACH DETECTOR", fontsize=16, weight="bold")\n    ax.text(592, 628, "Loud", fontsize=20, weight="bold", ha="center")\n    ax.text(870, 628, "Spread out", fontsize=20, weight="bold", ha="center")\n    ax.text(592, 655, "1 device · 20 min", fontsize=13, color="#B5B3C8", ha="center")\n    ax.text(870, 655, "80 devices · 3 h", fontsize=13, color="#B5B3C8", ha="center")\n    for idx, (detector, name, desc) in enumerate([\n        ("toy_model", "Toy transaction model*", "Amount · hour · distance · card history"),\n        ("device_rule", "Per-device velocity", "20+ events / 15+ cards in 5 min"),\n        ("checkout_rule", "Checkout-wide rule", "Small payments + card saves in 5 min"),\n    ]):\n        y = 694 + idx * 113\n        ax.add_patch(FancyBboxPatch((64, y), 952, 97, boxstyle="round,pad=0,rounding_size=12", facecolor=panel, edgecolor="none"))\n        ax.text(84, y + 37, name, fontsize=17, weight="bold")\n        ax.text(84, y + 66, desc, fontsize=12, color="#B5B3C8")\n        for x, scenario in [(592, "loud"), (870, "spread")]:\n            r = summary[(summary.scenario == scenario) & (summary.detector == detector)].iloc[0]\n            ax.text(x, y + 44, f"{r.coverage:.0%}", fontsize=30, weight="bold", ha="center", color=green if r.coverage > .5 else "#FF7979")\n            ax.text(x, y + 73, f"{r.flagged_events} / {r.events} events", fontsize=12, color="#B5B3C8", ha="center")\n    fps = summary[summary.scenario.str.startswith("legitimate")].groupby("detector").flagged_events.sum()\n    benign_total = int(summary[(summary.scenario.str.startswith("legitimate")) & (summary.detector == "device_rule")].events.sum())\n    ax.text(64, 1080, "Coverage comes with false positives.", fontsize=23, weight="bold")\n    ax.text(64, 1119, f"Of {benign_total:,} legitimate events: device flagged {fps[\'device_rule\']} · checkout {fps[\'checkout_rule\']}", fontsize=17, color="#D1D0DF")\n    ax.text(64, 1173, "*Payments only; 160 attack card-save events are outside model scope.\\nCoverage is retrospective within fixed UTC bins, not real-time prevention.\\nInvented data and scenarios. These are not production performance claims.", fontsize=13, color="#B5B3C8", linespacing=1.65, va="top")\n    ax.text(64, 1294, "Measure the behavior your detector can actually see.", fontsize=18, weight="bold", color="#ABA2FF")\n    fig.savefig(path, dpi=100, facecolor=bg)\n    plt.close(fig)\n\n\ndef build_metrics(args: argparse.Namespace, events: pd.DataFrame, csv_text: str, classic: dict,\n                  summary: pd.DataFrame) -> dict:\n    benign = events.label == 0\n    return {"seed": args.seed, "spread_devices": args.spread_devices, "event_count": len(events),\n            "event_sha256": hashlib.sha256(csv_text.encode()).hexdigest(), "classic_test": classic,\n            "synthetic_limitations": "Generators encode assumptions. No real fraud benchmark, generalization, production detection, or prevention claims.",\n            "coverage_definition": "Flagged events / scenario events. Fixed UTC five-minute bins, assigned retrospectively. Model scores payments only; unscored setups are uncovered.",\n            "currency_and_units": "USD integer minor units; small payment means <=100 cents ($1).",\n            "benign_false_positives": {d: int(events.loc[benign, d].sum()) for d in DETECTORS},\n            "benign_event_count": int(benign.sum()),\n            "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__, "matplotlib": matplotlib.__version__},\n            "scenarios": json.loads(summary.to_json(orient="records")),\n            "self_checks": ["unique invented event IDs and card identifiers", "timestamps round-trip in UTC", "disjoint stratified training/validation/test sample IDs", "no label/scenario/ID in features", "repeat generation identical", "independent rule aggregation agrees"]}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--output-dir", "--outdir", dest="output_dir", default="lab_output")\n    p.add_argument("--seed", type=int, default=SEED)\n    p.add_argument("--spread-devices", type=int, default=80)\n    args = p.parse_args()\n    out = Path(args.output_dir); out.mkdir(parents=True, exist_ok=True)\n    model, threshold, classic, splits = train_model(args.seed)\n    events = generate_events(args.seed, args.spread_devices)\n    assert events.equals(generate_events(args.seed, args.spread_devices)), "Generator is not reproducible"\n    events = apply_detectors(events, model, threshold)\n    reference_rule_flags(events)\n    summary = summarize(events)\n    csv_text = events.to_csv(index=False, float_format="%.10f", lineterminator="\\n")\n    # Write the exact bytes fingerprinted below, independent of host newline conventions.\n    (out / "card_testing_events.csv").write_bytes(csv_text.encode("utf-8"))\n    summary.to_csv(out / "card_testing_metrics.csv", index=False, float_format="%.10f",\n                   lineterminator="\\n", encoding="utf-8")\n    splits.to_csv(out / "classic_split_manifest.csv", index=False, lineterminator="\\n", encoding="utf-8")\n    metrics = build_metrics(args, events, csv_text, classic, summary)\n    (out / "card_testing_metrics.json").write_bytes((json.dumps(metrics, indent=2, allow_nan=False) + "\\n").encode("utf-8"))\n    # The official chart is for the default 80-device baseline. Avoid mislabeling a variant.\n    chart_path = out / "card_testing_chart.png"\n    if args.spread_devices == 80:\n        draw_chart(chart_path, summary, classic)\n    else:\n        chart_path.unlink(missing_ok=True)\n    print(json.dumps({k: metrics[k] for k in ["seed", "event_count", "event_sha256", "classic_test", "benign_false_positives"]}, indent=2))\n    print(summary[["scenario", "detector", "events", "flagged_events", "coverage"]].to_string(index=False))\n    print(f"Saved verified outputs to: {out.resolve()}")\n\n\nif __name__ == "__main__":\n    main()\n'
Path('card_testing_lab.py').write_text(SOURCE)
SEED = 61027
SPREAD_DEVICES = 80
OUT = Path(f'lab_output_seed{SEED}_devices{SPREAD_DEVICES}')
subprocess.run([sys.executable, 'card_testing_lab.py', '--seed', str(SEED), '--spread-devices', str(SPREAD_DEVICES), '--output-dir', str(OUT)], check=True)


## Inspect the measured results

`coverage` is flagged events divided by all events in that scenario. For the model, consult `payment_coverage` alongside `model_unscored_events`. The classic-fraud test set is independent from the four-hour event stream and uses a different synthetic population.


In [ ]:
import json
from IPython.display import Image, display
import pandas as pd

metrics = json.loads((OUT / "card_testing_metrics.json").read_text())
summary = pd.read_csv(OUT / "card_testing_metrics.csv")
print("Classic held-out test:")
display(pd.DataFrame([metrics["classic_test"]])[["precision", "recall", "false_positive_rate", "tp", "fp", "fn", "tn", "threshold"]])
print("Retrospective event coverage, all events:")
display(summary[["scenario", "detector", "events", "flagged_events", "coverage"]])
print("Transaction model, payments only; card saves remain unscored:")
display(summary[summary.detector == "toy_model"][["scenario", "payment_events", "flagged_payment_events", "payment_coverage", "model_unscored_events"]])
print("Legitimate events flagged:", metrics["benign_false_positives"], "out of", metrics["benign_event_count"])
print("Reproducibility fingerprint:", metrics["event_sha256"])
if (OUT / "card_testing_chart.png").exists():
    display(Image(filename=str(OUT / "card_testing_chart.png")))


## Verify reproducibility and the export

The script checks unique event IDs, UTC timestamps, isolated data splits, feature exclusions, repeat generation, and rule agreement with an independent aggregation. This additional cell reruns the entire pipeline into another directory and requires an identical event CSV fingerprint, including model scores and all detector flags. Nothing about these checks establishes production performance.


In [ ]:
import hashlib
import tempfile

with tempfile.TemporaryDirectory(prefix="card_lab_repeat_") as repeat_dir:
    subprocess.run([sys.executable, "card_testing_lab.py", "--seed", str(SEED), "--spread-devices", str(SPREAD_DEVICES), "--output-dir", repeat_dir], check=True, stdout=subprocess.DEVNULL)
    repeat_metrics = json.loads((Path(repeat_dir) / "card_testing_metrics.json").read_text())
    assert repeat_metrics["event_sha256"] == metrics["event_sha256"], "Repeat output changed"
events = pd.read_csv(OUT / "card_testing_events.csv")
assert len(events) == metrics["event_count"] == events.event_id.nunique()
assert hashlib.sha256((OUT / "card_testing_events.csv").read_bytes()).hexdigest() == metrics["event_sha256"]
print("PASS: full rerun is identical; export has", len(events), "unique events.")
print("PASS:", "; ".join(metrics["self_checks"]))


## Splunk handoff

Load the generated `card_testing_events.csv` into Splunk once, into its own index, `card_testing_lab`. The ZIP from the last cell contains it; Colab runs on Google's servers, so download it to your computer first.

- **With the lab repository:** run `python3 splunk/start_lab.py --csv path/to/card_testing_events.csv`. It indexes the file once and takes each event's time from the `epoch` column.
- **Manual upload (for example, Splunk Cloud):** Settings → Add Data → Upload, source type `csv`, then Timestamp → Advanced with timestamp field `epoch` and format `%s`, and the index `card_testing_lab`. Upload only once: duplicates inflate every count.

Set the search time range to **All time**; the events are dated 26 September 2026 (UTC). Check ingestion first: the event count and the unique `event_id` count must both equal the CSV's row count, with zero timestamp error.

The next cell writes the exact searches to `card_testing_rules.spl`. Run one search at a time, without the `/* ... */` comments. The parity search must return zero mismatches for all three rules. The model flags are precomputed Python results, not a model trained inside Splunk.

`strftime` follows Splunk's user display timezone. Set it to UTC for a readable UTC label; the numeric `bucket_epoch` is authoritative regardless of display timezone.

In [ ]:
RULES = '/* Import card_testing_events.csv into an isolated index named card_testing_lab.\n   Set search time range to All time. All rule criteria below are recomputed;\n   imported Python flags are used only by the final parity check.\n   Fixed UTC bins; retrospective flagged-event coverage, not prevention latency.\n*/\n\n/* D1: per-device counts; all payment and card-save events, approvals included. */\nindex=card_testing_lab event_source="synthetic_lab"\n| eval epoch=tonumber(epoch), bucket_epoch=floor(epoch/300)*300\n| stats count AS attempts dc(card_fp) AS cards count(eval(outcome="declined")) AS declines BY checkout_id device_fp bucket_epoch\n| eval decline_share=declines/attempts\n| where attempts>=20 AND cards>=15\n| eval bucket_display=strftime(bucket_epoch,"%Y-%m-%d %H:%M:%S")\n| table checkout_id device_fp bucket_epoch bucket_display attempts cards declines decline_share\n\n/* D2: candidate events only; low-value payments (USD cents) OR card saves. */\nindex=card_testing_lab event_source="synthetic_lab"\n| eval epoch=tonumber(epoch), amount_minor=tonumber(amount_minor), bucket_epoch=floor(epoch/300)*300\n| where flow="setup_card" OR (flow="payment" AND currency="USD" AND amount_minor<=100)\n| stats count AS candidates dc(card_fp) AS cards BY checkout_id bucket_epoch\n| where candidates>=20 AND cards>=15\n| table checkout_id bucket_epoch candidates cards\n\n/* Full event-level parity check; all three mismatch totals must be zero.\n   Labels and scenarios are not used by the detection criteria. */\nindex=card_testing_lab event_source="synthetic_lab"\n| eval epoch=tonumber(epoch), amount_minor=tonumber(amount_minor), bucket_epoch=floor(epoch/300)*300\n| eval candidate_spl=if(flow="setup_card" OR (flow="payment" AND currency="USD" AND amount_minor<=100),1,0)\n| eventstats count AS d_attempts dc(card_fp) AS d_cards count(eval(outcome="declined")) AS d_declines BY checkout_id device_fp bucket_epoch\n| eventstats sum(candidate_spl) AS c_attempts dc(eval(if(candidate_spl=1,card_fp,null()))) AS c_cards BY checkout_id bucket_epoch\n| eval device_spl=if(d_attempts>=20 AND d_cards>=15,1,0)\n| eval checkout_spl=if(candidate_spl=1 AND c_attempts>=20 AND c_cards>=15,1,0)\n| eval decline_spl=if(d_attempts>=20 AND d_cards>=15 AND d_declines/d_attempts>=0.8,1,0)\n| eval device_mismatch=if(device_spl=tonumber(device_rule),0,1), checkout_mismatch=if(checkout_spl=tonumber(checkout_rule),0,1), decline_mismatch=if(decline_spl=tonumber(decline_rule),0,1)\n| stats count AS ingested_events dc(event_id) AS unique_events sum(device_mismatch) AS device_mismatches sum(checkout_mismatch) AS checkout_mismatches sum(decline_mismatch) AS decline_mismatches\n\n/* Scenario coverage using recomputed flags. Labels/scenarios appear only here,\n   after the actual detection criteria, to measure known synthetic ground truth. */\nindex=card_testing_lab event_source="synthetic_lab"\n| eval epoch=tonumber(epoch), amount_minor=tonumber(amount_minor), bucket_epoch=floor(epoch/300)*300\n| eval candidate_spl=if(flow="setup_card" OR (flow="payment" AND currency="USD" AND amount_minor<=100),1,0)\n| eventstats count AS d_attempts dc(card_fp) AS d_cards count(eval(outcome="declined")) AS d_declines BY checkout_id device_fp bucket_epoch\n| eventstats sum(candidate_spl) AS c_attempts dc(eval(if(candidate_spl=1,card_fp,null()))) AS c_cards BY checkout_id bucket_epoch\n| eval device_spl=if(d_attempts>=20 AND d_cards>=15,1,0), checkout_spl=if(candidate_spl=1 AND c_attempts>=20 AND c_cards>=15,1,0), decline_spl=if(d_attempts>=20 AND d_cards>=15 AND d_declines/d_attempts>=0.8,1,0)\n| stats count AS events sum(device_spl) AS device_flagged sum(checkout_spl) AS checkout_flagged sum(decline_spl) AS decline_flagged sum(toy_model) AS model_flagged sum(toy_model_scored) AS model_scored BY scenario\n| eval device_coverage=round(100*device_flagged/events,2), checkout_coverage=round(100*checkout_flagged/events,2)\n\n/* NOTE: strftime uses the Splunk user\'s display timezone. Set user Preferences\n   timezone to UTC if a UTC bucket_display is desired; bucket_epoch remains authoritative. */\n'
(OUT / 'card_testing_rules.spl').write_text(RULES)
print(RULES)


## Save the lab outputs

The ZIP includes the events, measured metrics, chart, split manifest and Splunk searches. The notebook prints its location and a local download link. In hosted Colab, download the ZIP from the Files panel, or set `DOWNLOAD_IN_COLAB = True` and rerun this cell. It defaults to false so that **Run all** doesn't open a download dialog.

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Saved:", Path(archive).resolve())
display(FileLink(archive))
DOWNLOAD_IN_COLAB = False
if DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files
    except ImportError:
        print("This is a local runtime; use the download link above.")
    else:
        files.download(archive)
